# 04 — Interpretability with SHAPWe refit the best model (Random Forest) on the full data and use SHAP tounderstand what drives its predictions, both globally (which featuresmatter) and locally (why it predicted success or failure for a givenfounder).

In [ ]:
import sys, ossys.path.insert(0, os.path.abspath('../src'))import pandas as pdimport matplotlib.pyplot as pltfrom features import build_features, feature_columnsfrom shap_analysis import (fit_best_model, compute_shap_values,                            plot_shap_summary, plot_shap_bar,                            plot_shap_dependence)import shapdf = pd.read_csv('../data/vcbench_final_public.csv')X_full = build_features(df)feats = feature_columns(X_full)X = X_full[feats]; y = X_full['success']print('Fitting best model on full data...')model = fit_best_model(X, y, seed=42)print('Computing SHAP values...')shap_values, X_sample, idx, names = compute_shap_values(model, X, max_samples=2000)print(f'SHAP values shape: {shap_values.shape}')

## Global feature importance (mean |SHAP|)

In [ ]:
import numpy as npmean_abs = np.abs(shap_values).mean(axis=0)imp = pd.DataFrame({'feature': names, 'mean_abs_shap': mean_abs})imp = imp.sort_values('mean_abs_shap', ascending=False).head(15)imp.round(4)

## SHAP summary plot

In [ ]:
shap.summary_plot(shap_values, X_sample, feature_names=names,                  max_display=15, show=True, plot_size=(10,7))

**Reading the plot.** Each row is one feature, each dot one founder.- X-axis: SHAP value (impact on predicted success probability).- Color: feature value (red = high, blue = low).For example, on `best_qs_ranking`: high values (red, i.e. low-prestigeuniversities) push predictions strongly negative; low values (blue, i.e.elite universities) push predictions positive. This is the strongest singledriver in the model.

## Dependence plots: top features in detail

In [ ]:
top_feats = list(imp['feature'].values[:4])print(f'Plotting dependence for: {top_feats}')for feat in top_feats:    fig, ax = plt.subplots(figsize=(7, 4))    feat_idx = names.index(feat)    ax.scatter(X_sample[:, feat_idx] if hasattr(X_sample,'shape') and X_sample.ndim==2               else X_sample[feat],               shap_values[:, feat_idx],               alpha=0.4, s=20, color='#264653')    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')    ax.set_xlabel(feat)    ax.set_ylabel('SHAP value')    ax.set_title(f'How {feat} affects the prediction')    plt.tight_layout()    plt.show()

## Wrap-upThe model has learned a story consistent with venture capital folk wisdom:elite education + prior exits + relevant industry experience + exposure tolarge organisations correlate with founder success. Crucially, this storyis **fully transparent** — we can audit, contest, and refine each piece ofit, which is impossible with a black-box LLM.